# 02 · Experiment template (one idea per notebook)

Copy to `experiments/<topic>/<NN>_<idea>.ipynb`.
Rules: same spec, same splits, same metric as every other run; **one change vs. the baseline**;
save the run (so it can be blended) and report CV mean ± std against the baseline run.

In [ ]:
%load_ext autoreload
%autoreload 2
import os
# os.environ["QRTC_CONFIG"] = "experiments/intraday_ridge/spec.toml"  # another spec
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.max_columns", 60, "display.width", 160)
from qrtc import config, io, eda, features as F, cv, train, tracking, ensemble, postprocess, submit
from qrtc.metrics import get_metric
spec = config.load_spec()
spec

In [ ]:
IDEA = "describe-the-idea"        # becomes part of the run_id
MODEL = {"survival": "xgb"}.get(spec.task, "lgbm")  # presets: lgbm_shallow, xgb_shallow, ...
PARAMS = {}
ds = io.load_dataset(spec)

## Features: pure function train/test → new frames (never mutate inputs)

In [ ]:
def make_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df
    ret = F.col_block(df, "RET")
    if spec.date_col and ret:
        out = F.cs_rank(out, ret[:5], spec.date_col)                    # per-date ranks
        out = F.window_sums(out, ret, [3, 5, 10, 20], "RET")              # multi-day returns
        out = F.row_stats(out, ret, "RET")
        cats = [c for c in ("SECTOR", "GROUP", "INDUSTRY") if c in df]
        if cats:
            out = F.group_demean(out, ret[:5], [spec.date_col, cats[0]])  # remove sector move
            out = F.group_agg(out, ret[:5], [spec.date_col], ("mean",))   # market move
    return out

tr, te = make_features(ds.train), make_features(ds.test)
feats = io.feature_columns(tr, spec)
X, X_test = F.to_category(tr[feats], te[feats], F.auto_categoricals(tr[feats]))
X.shape

## CV (identical splits for every run: same spec, same seed)

In [ ]:
splits = cv.make_splits(ds.train, ds.y_vector, spec, test=ds.test)
groups = ds.train[spec.date_col].to_numpy() if spec.date_col else None
weight = io.train_weight(ds, spec)
res = train.run_cv(spec, X, ds.y_vector, X_test, splits, MODEL, PARAMS, weight=weight, groups=groups)

In [ ]:
res.importance.head(25)

## Compare with the run leaderboard, then save

In [ ]:
tracking.leaderboard(competition=spec.name).head(10)

In [ ]:
run_id = tracking.save_run(res, spec, IDEA, ds.train[spec.id_col].to_numpy(),
                           ds.test[spec.id_col].to_numpy(), notes=IDEA)
run_id

## Optional: submission from this run

In [ ]:
# test_ids = ds.test[spec.id_col].to_numpy()
# sub = submit.build_submission(spec, test_ids, res.test_pred, ds.classes)
# submit.write_submission(sub, spec, test_ids, run_id)